In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

df = pd.read_csv('../data/raw/fake_job_postings.csv')
stop_words = set(stopwords.words('english'))

print("Data loaded!")
print("Shape:", df.shape)

Data loaded!
Shape: (17880, 18)


In [2]:
# Step 1 - Combine all text columns
df['combined_text'] = (
    df['title'].fillna('') + ' ' +
    df['company_profile'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['requirements'].fillna('')
)

# Step 2 - Clean text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join([w for w in text.split()
                     if w not in stop_words])
    return text

df['clean_text'] = df['combined_text'].apply(clean_text)

print("Text cleaning done!")
print("Sample cleaned text:")
print(df['clean_text'].iloc[0][:300])

Text cleaning done!
Sample cleaned text:
marketing intern food weve created groundbreaking awardwinning cooking site support connect celebrate home cooks give everything need one placewe top editorial business engineering team focused using technology find new better ways connect people around specific food interests offer superb highly cu


In [3]:
# Step 3 - Engineer numeric features
df['desc_length'] = df['description'].fillna('').apply(len)
df['desc_word_count'] = df['description'].fillna('').apply(
    lambda x: len(x.split()))
df['has_company_profile'] = df['company_profile'].notna().astype(int)
df['has_requirements'] = df['requirements'].notna().astype(int)
df['has_benefits'] = df['benefits'].notna().astype(int)
df['has_salary'] = df['salary_range'].notna().astype(int)

# Suspicious keyword counter
suspicious = ['urgent','guaranteed','no experience',
              'work from home','earn money',
              'unlimited income','be your own boss',
              'weekly pay','immediate start']
df['suspicious_word_count'] = df['combined_text'].apply(
    lambda x: sum(1 for w in suspicious if w in x.lower())
)

# Employment type encoding
df['is_parttime'] = (
    df['employment_type'] == 'Part-time').astype(int)

NUMERIC_FEATURES = [
    'desc_length','desc_word_count',
    'has_company_profile','has_requirements',
    'has_benefits','has_salary',
    'suspicious_word_count','is_parttime'
]

print("Numeric features done!")
print(df[NUMERIC_FEATURES].describe().round(2))

Numeric features done!
       desc_length  desc_word_count  has_company_profile  has_requirements  \
count     17880.00         17880.00             17880.00          17880.00   
mean       1218.00           170.45                 0.81              0.85   
std         894.83           123.30                 0.39              0.36   
min           0.00             0.00                 0.00              0.00   
25%         607.00            87.00                 1.00              1.00   
50%        1017.00           146.00                 1.00              1.00   
75%        1586.00           224.00                 1.00              1.00   
max       14907.00          2115.00                 1.00              1.00   

       has_benefits  has_salary  suspicious_word_count  is_parttime  
count      17880.00    17880.00               17880.00     17880.00  
mean           0.60        0.16                   0.05         0.04  
std            0.49        0.37                   0.24         0

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

# TF-IDF on cleaned text
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)
text_features = tfidf.fit_transform(df['clean_text'])

# Combine text + numeric features
numeric_matrix = sp.csr_matrix(df[NUMERIC_FEATURES].values)
X = sp.hstack([text_features, numeric_matrix])
y = df['fraudulent']

print("TF-IDF shape:", text_features.shape)
print("Combined X shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

TF-IDF shape: (17880, 5000)
Combined X shape: (17880, 5008)
Target distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [5]:
import pickle

# Save TF-IDF vectorizer
with open('../src/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save numeric feature names
with open('../src/numeric_features.pkl', 'wb') as f:
    pickle.dump(NUMERIC_FEATURES, f)

# Save processed dataframe
df.to_csv('../data/processed/model_data.csv', index=False)

print("All saved successfully!")
print("Files saved:")
print("  src/tfidf.pkl")
print("  src/numeric_features.pkl")
print("  data/processed/model_data.csv")

All saved successfully!
Files saved:
  src/tfidf.pkl
  src/numeric_features.pkl
  data/processed/model_data.csv
